# Day 6: Data Quality, Validation & Quarantine DLQ Engine

This notebook demonstrates using PyDeequ for structural rules and clinical plausibility with Dead Letter Queue (DLQ) routing.

In [ ]:
!pip install pyspark pyiceberg duckdb splink pydeequ -q

In [ ]:
import os
from pyspark.sql import SparkSession

# Set required SPARK_VERSION env var for PyDeequ
os.environ["SPARK_VERSION"] = "3.5"

spark = SparkSession.builder \
    .appName("Day06_Data_Quality") \
    .config("spark.jars.packages", "com.amazon.deequ:deequ:2.0.7-spark-3.3") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark session with PyDeequ initialized.")

In [ ]:
from pyspark.sql.functions import col
import pydeequ
from pydeequ.checks import Check, CheckLevel
from pydeequ.verification import VerificationSuite

# Create synthetic OMOP data with intentional bad records
data = [
    ("cond_1", "person_123", 31967),  # Valid
    ("cond_2", None, 31967),          # Invalid: missing person_id
    ("cond_3", "person_456", -100)    # Invalid: negative concept_id
]

df = spark.createDataFrame(data, ["condition_occurrence_id", "person_id", "condition_concept_id"])
df.show()

In [ ]:
# Define and run PyDeequ checks
check = Check(spark, CheckLevel.Error, "OMOP_Silver_DQ_Check")

checkResult = VerificationSuite(spark) \
    .onData(df) \
    .addCheck(
        check.isComplete("person_id")
        .isComplete("condition_concept_id")
        .isNonNegative("condition_concept_id")
    ) \
    .run()

checkResult_df = VerificationSuite.checkResultsAsDataFrame(spark, checkResult)
print("Data Quality Verification Results:")
checkResult_df.show(truncate=False)

In [ ]:
# Simulate row-level DLQ routing based on the rules
valid_df = df.filter(col("person_id").isNotNull() & (col("condition_concept_id") >= 0))
dlq_df = df.filter(col("person_id").isNull() | (col("condition_concept_id") < 0))

print("VALID RECORDS (Routed to Silver Valid):")
valid_df.show()

print("QUARANTINED RECORDS (Routed to DLQ):")
dlq_df.show()